# Mechanism analysis: first/last anchoring? (read-only, S=7, seed 42)

Notebook 37 showed a striking pattern: models trained on S7_k02 execute a singleton TWICE (A → AA, B → BB,
C → CC = unchanged), and on k=3 they return a two-operation result. Hypothesis: with short training instructions,
the model organises the computation around the FIRST and the LAST real operation. A singleton has first = last
(executed twice), and a longer instruction loses its middle operation(s).

Two read-only checks; nothing is trained:
1. **Which operation is dropped** (from the saved predictions): for held-out longer instructions whose prediction
   equals the word with one operation removed, the dropped position. The hypothesis predicts MIDDLE.
2. **Attention probe** (loads the LAST checkpoints): how much attention the answer positions and the input cells pay
   to the FIRST / MIDDLE / LAST real operation and to the `<identity>` fillers, per layer and per head. This covers
   S7_k02, S7_k0234 and S7_k02345 (the success reference), for AR34, query AR, causal encoder and original AR.

A CPU runtime works but is slow; a GPU makes the probe much faster (set `DEVICE = "cuda"`).
**Send back `Shortcut_Analysis_S7_short_k_Review.ipynb` and `Attention_Probe_Review.ipynb`.**


In [1]:
# --- 2. Bootstrap: locate or clone the PRIVATE repository (credentials are never printed or stored) ---
import base64, importlib.util, os, pathlib, subprocess, sys

REPO_URL = "https://github.com/zhexuandliu/Decompositionality.git"
REPO_BRANCH = "main"
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

def _find_repo():
    candidates = [pathlib.Path("/content/Decompositionality"), pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    for p in candidates:
        if (p / "src" / "decompositionality").is_dir():
            return p.resolve()
    return None

REPO_DIR = _find_repo()
if REPO_DIR is None:
    token = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
    if not token:
        import getpass
        token = getpass.getpass("GitHub token with read access to the private repo (input hidden): ")
    # Pass the credential through environment-scoped git config (not the URL, not the command line,
    # not persisted in .git/config).
    _hdr = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()
    _env = dict(os.environ, GIT_TERMINAL_PROMPT="0", GIT_CONFIG_COUNT="1",
                GIT_CONFIG_KEY_0="http.https://github.com/.extraheader", GIT_CONFIG_VALUE_0=_hdr)
    REPO_DIR = pathlib.Path("/content/Decompositionality" if IN_COLAB else "Decompositionality").resolve()
    _r = subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
                        env=_env, capture_output=True, text=True)
    del token, _hdr, _env
    if _r.returncode != 0:
        raise RuntimeError("git clone failed (check the GITHUB_TOKEN secret / repo access):\n" + _r.stderr[-2000:])

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
for _mod in ("numpy", "pandas", "matplotlib", "torch"):
    if importlib.util.find_spec(_mod) is None:  # Colab already ships these; never reinstall torch/CUDA here
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _mod], check=True)
REPO_COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Repository:", REPO_DIR, "| commit:", REPO_COMMIT or "unknown")


Repository: /content/Decompositionality | commit: f428bcd63a14793a7cec0be814c1eb0ca344c761


In [2]:
# --- CONFIGURATION -------------------------------------------------------------------------------------------------
MODE = "full"
SEED = 42
USE_DRIVE = IN_COLAB
RESULTS_ROOT = os.environ.get("DECOMP_RESULTS_ROOT") or (
    "/content/drive/MyDrive/decompositionality_results" if USE_DRIVE else str(REPO_DIR / "results"))
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# -----------------------------------------------------------------------------------------------------------------
if USE_DRIVE and IN_COLAB and RESULTS_ROOT.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
import pandas as pd
pd.set_option("display.max_rows", 500); pd.set_option("display.max_columns", 60); pd.set_option("display.width", 250)
from decompositionality import shortcut_analysis as SA, attn_probe as AP
print("device:", DEVICE)


Mounted at /content/drive
device: cuda


In [3]:
# --- 1. Predictions: outcome classes + which operation is dropped (table 5) ---------------------------------------
rp1 = SA.analyse(RESULTS_ROOT, MODE, SEED)
print("REVIEW FILE TO SEND BACK:", rp1)
if IN_COLAB and rp1:
    from google.colab import files
    files.download(str(rp1))


# What do the models compute on held-out inputs? (seed 42, LAST checkpoint)
Read-only: rebuilt sampled test sets are checked record-by-record against the saved predictions. A transform is written as its shortest A/B/C word (A = right shift, B = clockwise rotation, C = left-right mirror; executed left to right). 'seen' = the transform is the overall transform of some trained instruction.


### Availability


,condition,model,status
0,S7_k02,original AR,ok
1,S7_k02,AR34,ok
2,S7_k02,query AR,ok
3,S7_k02,causal encoder,ok
4,S7_k023,original AR,ok
5,S7_k023,AR34,ok
6,S7_k023,query AR,ok
7,S7_k023,causal encoder,ok
8,S7_k0234,original AR,ok
9,S7_k0234,AR34,ok


**Which operation is dropped?** For held-out longer instructions whose prediction equals the word with exactly ONE operation removed, the position (1 = first, k = last) of that operation. 'unique' counts only records where exactly one position fits; 'any' counts every fitting position.


### 5. First held-out longer k: which operation is dropped (first 60 of 72 rows; full table in CSV export)


,condition,model,k,dropped position,role,unique,any,share of unique drops
0,S7_k02,original AR,3,1,first,1,6667,0.001
1,S7_k02,original AR,3,2,middle,778,11889,0.999
2,S7_k02,original AR,3,3,last,0,6667,0.000
3,S7_k02,AR34,3,1,first,709,7375,0.089
4,S7_k02,AR34,3,2,middle,1386,12497,0.174
5,S7_k02,AR34,3,3,last,5880,12547,0.737
6,S7_k02,query AR,3,1,first,15,6681,0.011
7,S7_k02,query AR,3,2,middle,1316,12427,0.989
8,S7_k02,query AR,3,3,last,0,6667,0.000
9,S7_k02,causal encoder,3,1,first,0,6666,0.000


... (9 outputs left out)

,condition,model,k,records,correct,identity copy,= one op dropped,of which = last op dropped,other legal transform,no legal / invalid
0,S7_k02,original AR,3,20000,0.000,0.037,0.594,0.333,0.557,0.406
1,S7_k02,AR34,3,20000,0.000,0.037,0.954,0.627,0.954,0.009
2,S7_k02,query AR,3,20000,0.000,0.037,0.622,0.333,0.585,0.378
3,S7_k02,causal encoder,3,20000,0.000,0.037,0.665,0.401,0.628,0.335
4,S7_k023,original AR,4,20000,0.000,0.000,0.157,0.059,0.157,0.843
5,S7_k023,AR34,4,20000,0.000,0.000,0.447,0.135,0.447,0.553
6,S7_k023,query AR,4,20000,0.000,0.000,1.000,0.333,1.000,0.000
7,S7_k023,causal encoder,4,20000,0.000,0.000,0.342,0.141,0.342,0.658
8,S7_k0234,original AR,5,20000,0.000,0.052,0.660,0.273,0.608,0.340
9,S7_k0234,AR34,5,20000,0.000,0.021,0.165,0.036,0.144,0.835


REVIEW FILE: /content/drive/MyDrive/decompositionality_results/S7_support_sweep_20k_v1/report/seed_42_shortcut_analysis/Shortcut_Analysis_S7_short_k_Review.ipynb
REVIEW FILE TO SEND BACK: /content/drive/MyDrive/decompositionality_results/S7_support_sweep_20k_v1/report/seed_42_shortcut_analysis/Shortcut_Analysis_S7_short_k_Review.ipynb


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
# --- 2. Attention probe (loads the LAST checkpoints) ---------------------------------------------------------------
rp2 = AP.probe(RESULTS_ROOT, MODE, SEED, device=DEVICE)
print("REVIEW FILE TO SEND BACK:", rp2)
if IN_COLAB and rp2:
    from google.colab import files
    files.download(str(rp2))


# Attention to the operation tokens (LAST checkpoint, teacher forcing, fresh records)
Roles: FIRST / MIDDLE / LAST real operation and `<identity>` fillers (a singleton's one operation counts as FIRST). 'op mass' = the share of a query position's attention that lands on the 7 operation tokens; the role shares split that op mass; 'uniform ...' = the share each role would get if attention were spread evenly over the 7 slots (the baseline to compare with). Averaged over heads and query positions (layer table) or over query positions (head table).


### Availability


,condition,model,status
0,S7_k02,AR34,ok (step 20000)
1,S7_k02,query AR,ok (step 20000)
2,S7_k02,causal encoder,ok (step 20000)
3,S7_k02,original AR,ok (step 20000)
4,S7_k0234,AR34,ok (step 20000)
5,S7_k0234,query AR,ok (step 20000)
6,S7_k0234,causal encoder,ok (step 20000)
7,S7_k0234,original AR,ok (step 20000)
8,S7_k02345,AR34,ok (step 20000)
9,S7_k02345,query AR,ok (step 20000)



### 1. Per layer (heads weighted by their op mass): op mass and its split over roles


,condition,model,records,query,layer,op mass,FIRST,MIDDLE,LAST,<identity>,uniform FIRST,uniform MIDDLE,uniform LAST,uniform <identity>
0,S7_k02,AR34,singleton,answer positions,1,0.364,0.975,0.000,0.000,0.025,0.143,0.000,0.000,0.857
1,S7_k02,AR34,singleton,input cells,1,0.503,0.774,0.000,0.000,0.226,0.143,0.000,0.000,0.857
2,S7_k02,AR34,singleton,answer positions,2,0.350,0.860,0.000,0.000,0.140,0.143,0.000,0.000,0.857
3,S7_k02,AR34,singleton,input cells,2,0.379,0.834,0.000,0.000,0.166,0.143,0.000,0.000,0.857
4,S7_k02,AR34,singleton,answer positions,3,0.237,0.450,0.000,0.000,0.550,0.143,0.000,0.000,0.857
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
523,S7_k02345,original AR,k=6 (held out),input cells,4,0.034,0.115,0.707,0.105,0.073,0.143,0.571,0.143,0.143
524,S7_k02345,original AR,k=6 (held out),answer positions,5,0.008,0.074,0.580,0.224,0.122,0.143,0.571,0.143,0.143
525,S7_k02345,original AR,k=6 (held out),input cells,5,0.019,0.057,0.633,0.210,0.099,0.143,0.571,0.143,0.143
526,S7_k02345,original AR,k=6 (held out),answer positions,6,0.025,0.106,0.579,0.194,0.121,0.143,0.571,0.143,0.143



### 2. Heads that put >= 10% of their attention on the operation tokens (trained-length records)


,condition,model,records,query,layer,head,op mass,FIRST,MIDDLE,LAST,<identity>,uniform FIRST,uniform MIDDLE,uniform LAST,uniform <identity>,FIRST - LAST
72,S7_k02,AR34,k=2 (trained),answer positions,1,1,1.000,0.350,0.000,0.650,0.000,0.143,0.000,0.143,0.714,-0.301
75,S7_k02,AR34,k=2 (trained),answer positions,1,4,0.210,0.419,0.000,0.380,0.201,0.143,0.000,0.143,0.714,0.040
77,S7_k02,AR34,k=2 (trained),answer positions,1,6,0.995,0.517,0.000,0.483,0.000,0.143,0.000,0.143,0.714,0.033
84,S7_k02,AR34,k=2 (trained),answer positions,2,1,0.366,0.448,0.000,0.504,0.048,0.143,0.000,0.143,0.714,-0.056
85,S7_k02,AR34,k=2 (trained),answer positions,2,2,0.343,0.203,0.000,0.614,0.183,0.143,0.000,0.143,0.714,-0.411
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2482,S7_k02345,query AR,k=5 (trained),input cells,3,5,0.139,0.082,0.466,0.173,0.279,0.143,0.429,0.143,0.286,-0.091
2421,S7_k02345,query AR,k=2 (trained),input cells,4,4,0.270,0.166,0.000,0.145,0.689,0.143,0.000,0.143,0.714,0.021
2493,S7_k02345,query AR,k=5 (trained),input cells,4,4,0.256,0.211,0.406,0.114,0.270,0.143,0.429,0.143,0.286,0.097
2422,S7_k02345,query AR,k=2 (trained),input cells,4,5,0.299,0.137,0.000,0.145,0.718,0.143,0.000,0.143,0.714,-0.008


REVIEW FILE: /content/drive/MyDrive/decompositionality_results/S7_support_sweep_20k_v1/report/seed_42_attention_probe/Attention_Probe_Review.ipynb
REVIEW FILE TO SEND BACK: /content/drive/MyDrive/decompositionality_results/S7_support_sweep_20k_v1/report/seed_42_attention_probe/Attention_Probe_Review.ipynb


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>